# Getting Started with Mascope SDK

This notebook walks through the basics: connecting to a Mascope server, browsing the data hierarchy, and plotting your first spectrum and peak data.


### Setup

Create a `.env` file in your working directory (or any parent directory) with your credentials:

```env
MASCOPE_URL=https://your-instance.mascope.app
MASCOPE_ACCESS_TOKEN=your-api-token
```

> **NOTE:** See the `README` for detailed configuration instructions


### Connect


In [ ]:
from mascope_sdk import MascopeClient


mascope = MascopeClient()
mascope

### Browse the data hierarchy

Mascope organises data as: **Datasets** → **Batches** → **Samples**.
All listing methods return pandas DataFrames.

> **Tip:** Name filters accept plain substrings _or_ regular expressions (case-insensitive).
> For example, `batches="^Uronium.*March$"` matches names starting with "Uronium" and ending with "March",
> while `samples="blank|control"` matches names containing either word.


In [ ]:
# List all datasets
datasets = mascope.datasets.list()
datasets

In [ ]:
# List batches in a dataset (by name, substring of name, or ID)
batches = mascope.batches.list("My Dataset")  # <- replace with your dataset name
batches

In [ ]:
# List samples in a batch
samples = mascope.samples.list(
    dataset="My Dataset", batch="My Batch"
)  # <- replace with your batch name
samples

In [ ]:
# List samples across multiple batches (filtered by a substring in the batch name)
samples = mascope.samples.list(
    dataset="My Dataset", batches="Batch"
)  # <- replace with a substring of your batch name
samples

### Plot a spectrum

Fetch the averaged mass spectrum of a sample and plot it.


In [ ]:
import matplotlib.pyplot as plt


sample_id = samples.iloc[0]["sample_item_id"]  # Pick the first sample in the list
spectrum = mascope.samples.get_spectrum(sample_id)

plt.figure(figsize=(12, 4))
plt.plot(spectrum["mz"], spectrum["intensity"])
plt.xlabel("m/z [Th]")
plt.ylabel("Intensity [cps]")
plt.title(f"Sum spectrum: {samples.iloc[0]['sample_item_name']}")
plt.tight_layout()
plt.show()

### Inspect peaks

Load peak data with compound match information.

**NOTE:** There may be more than one row per peak (`peak_id`) in case it is matched with more than one target isotope. If you need unique peaks only, you may use `peaks.drop_duplicates(subset=["peak_id"])` (see cells below). See also the other tutorial notebooks for more examples on aggregation of peak data, and selecting the best match for each peak.


In [ ]:
peaks = mascope.samples.get_peaks(sample_id)
peaks

In [ ]:
# Show only matched peaks (one row per match - a peak may appear multiple times)
matched = peaks[peaks["target_compound_formula"].notna()]
n_matched_peaks = matched["peak_id"].nunique()
n_total_peaks = peaks["peak_id"].nunique()
print(f"{n_matched_peaks} matched peaks out of {n_total_peaks} total")
matched[
    [
        "mz",
        "area",
        "height",
        "target_compound_name",
        "target_compound_formula",
        "target_ion_formula",
    ]
]

### Plot peaks


In [ ]:
# One row per physical peak for plotting (a peak may have multiple match rows)
unique_peaks = peaks.drop_duplicates(subset=["peak_id"])
unique_matched = unique_peaks[unique_peaks["target_compound_formula"].notna()]

plt.figure(figsize=(12, 4))
plt.scatter(unique_peaks["mz"], unique_peaks["mz"] - unique_peaks["mz"].round())
plt.scatter(
    unique_matched["mz"], unique_matched["mz"] - unique_matched["mz"].round()
)  # Highlight matched peaks
plt.xlabel("m/z [Th]")
plt.ylabel("Mass defect [Th]")
plt.title(f"Peaks: {samples.iloc[0]['sample_item_name']}")
plt.tight_layout()
plt.show()